In [1]:
# load snapshot data
import pandas as pd

snapshot = pd.read_csv(
    "../feature_store/student_snapshot.csv"
)  
behaviour = pd.read_csv(
    "../feature_store/behaviour_model_snapshot.csv"
)
academic= pd.read_csv(
    "../feature_store/academic_model_snapshot.csv"
) 

In [2]:
# target 
snapshot["grade_code_mode"].value_counts()

grade_code_mode
N            73
P            23
C            19
D            16
S            14
SA           13
HD            9
DX            3
No Result     1
Name: count, dtype: int64

In [3]:
# remove non-training columns 
snapshot = snapshot.drop(columns=["student_key", "course_key", "snapshot_date"])

In [4]:
# check how many top_material_type categories we have
snapshot["top_material_type"].value_counts()

top_material_type
Course Page         123
Assignment           35
Lecture Material     11
No Activity           2
Name: count, dtype: int64

In [5]:
# encoding catergorical features - one hot encoding
behaviour = pd.get_dummies(
    behaviour,
    columns=["top_material_type"],
    drop_first=False
)
behaviour.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 167 entries, 0 to 166
Data columns (total 14 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   student_key                         167 non-null    int64  
 1   course_key                          167 non-null    int64  
 2   snapshot_date                       167 non-null    object 
 3   total_events                        167 non-null    int64  
 4   active_days                         167 non-null    int64  
 5   unique_materials                    167 non-null    int64  
 6   after_hours_events                  167 non-null    int64  
 7   weekend_events                      167 non-null    int64  
 8   avg_events_per_active_day           167 non-null    float64
 9   risk_label                          167 non-null    int64  
 10  top_material_type_Assignment        167 non-null    bool   
 11  top_material_type_Course Page       167 non-n

In [6]:
# encoding academic model
academic = pd.get_dummies(
    academic,
    columns=["top_material_type"],
    drop_first=False
)
academic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 167 entries, 0 to 166
Data columns (total 32 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   student_key                         167 non-null    int64  
 1   course_key                          167 non-null    int64  
 2   total_events                        167 non-null    int64  
 3   active_days                         167 non-null    int64  
 4   unique_materials                    167 non-null    int64  
 5   after_hours_events                  167 non-null    int64  
 6   weekend_events                      167 non-null    int64  
 7   avg_events_per_active_day           167 non-null    float64
 8   snapshot_date                       167 non-null    object 
 9   risk_label                          167 non-null    int64  
 10  ass_1_with_penalty_100              167 non-null    float64
 11  ass_2_100                           167 non-n

In [7]:
!pip install scikit-learn

In [8]:
# behaviour train-test split
from sklearn.model_selection import train_test_split

X_behaviour = behaviour.drop(
    columns=[
        "student_key",
        "course_key",
        "snapshot_date",
        "risk_label"
    ]
)

y_behaviour = behaviour["risk_label"]

X_beh_train, X_beh_test, y_beh_train, y_beh_test = train_test_split(
    X_behaviour,
    y_behaviour,
    test_size=0.2,
    random_state=42,
    stratify=y_behaviour
)

# validation split (25% of train = 20% of original)
X_beh_train, X_beh_val, y_beh_train, y_beh_val = train_test_split(
    X_beh_train,
    y_beh_train,
    test_size=0.25,
    random_state=42,
    stratify=y_beh_train
)

In [ ]:
# academic train-test split
X_academic = academic.drop(
    columns=[
        "student_key",
        "course_key",
        "snapshot_date",
        "risk_label",

        # leakage columns
        "exam_100",
        "total_100",
        "rounded_total_100",

        # duplicate columns
        "assignment_1_100",
        "ass_2_100"
    ]
)

y_academic = academic["risk_label"]

X_aca_train, X_aca_test, y_aca_train, y_aca_test = train_test_split(
    X_academic,
    y_academic,
    test_size=0.2,
    random_state=42,
    stratify=y_academic
)

# validation split (25% of train = 20% of original)
X_aca_train, X_aca_val, y_aca_train, y_aca_val = train_test_split(
    X_aca_train,
    y_aca_train,
    test_size=0.25,
    random_state=42,
    stratify=y_aca_train
)

In [10]:
# save datasets
from pathlib import Path

dataset_dir = Path("../datasets")
dataset_dir.mkdir(exist_ok=True)

# behvaiour 
X_beh_train.to_csv(dataset_dir / "X_beh_train.csv", index=False)
X_beh_val.to_csv(dataset_dir / "X_beh_val.csv", index=False)
X_beh_test.to_csv(dataset_dir / "X_beh_test.csv", index=False)

y_beh_train.to_csv(dataset_dir / "y_beh_train.csv", index=False)
y_beh_val.to_csv(dataset_dir / "y_beh_val.csv", index=False)
y_beh_test.to_csv(dataset_dir / "y_beh_test.csv", index=False)

# academic
X_aca_train.to_csv(dataset_dir / "X_aca_train.csv", index=False)
X_aca_val.to_csv(dataset_dir / "X_aca_val.csv", index=False)
X_aca_test.to_csv(dataset_dir / "X_aca_test.csv", index=False)

y_aca_train.to_csv(dataset_dir / "y_aca_train.csv", index=False)
y_aca_val.to_csv(dataset_dir / "y_aca_val.csv", index=False)
y_aca_test.to_csv(dataset_dir / "y_aca_test.csv", index=False)

print(y_behaviour.value_counts())
print(y_academic.value_counts())

risk_label
1    100
0     67
Name: count, dtype: int64
risk_label
1    100
0     67
Name: count, dtype: int64


In [11]:
print(y_beh_train.value_counts(normalize=True))
print(y_beh_val.value_counts(normalize=True))
print(y_beh_test.value_counts(normalize=True))

risk_label
1    0.606061
0    0.393939
Name: proportion, dtype: float64
risk_label
1    0.588235
0    0.411765
Name: proportion, dtype: float64
risk_label
1    0.588235
0    0.411765
Name: proportion, dtype: float64


In [12]:
print(y_aca_train.value_counts(normalize=True))
print(y_aca_val.value_counts(normalize=True))
print(y_aca_test.value_counts(normalize=True))

risk_label
1    0.606061
0    0.393939
Name: proportion, dtype: float64
risk_label
1    0.588235
0    0.411765
Name: proportion, dtype: float64
risk_label
1    0.588235
0    0.411765
Name: proportion, dtype: float64


In [13]:
print(X_beh_train.shape)
print(X_beh_val.shape)
print(X_beh_test.shape)

(99, 10)
(34, 10)
(34, 10)


In [14]:
print(X_aca_train.shape)
print(X_aca_val.shape)
print(X_aca_test.shape)

(99, 23)
(34, 23)
(34, 23)


In [15]:
print(X_aca_train.columns.tolist())

['total_events', 'active_days', 'unique_materials', 'after_hours_events', 'weekend_events', 'avg_events_per_active_day', 'ass_1_with_penalty_100', 'ass_2_with_penalty_100', 'ass_ex_1', 'ass_ex_10', 'ass_ex_2', 'ass_ex_3', 'ass_ex_4', 'ass_ex_5', 'ass_ex_6', 'ass_ex_7', 'ass_ex_8', 'ass_ex_9', 'weekly_lab_quizzes_100', 'top_material_type_Assignment', 'top_material_type_Course Page', 'top_material_type_Lecture Material', 'top_material_type_No Activity']


In [16]:
X_aca_train.isnull().sum().sort_values(ascending=False).head(20)

total_events                          0
ass_ex_4                              0
top_material_type_Lecture Material    0
top_material_type_Course Page         0
top_material_type_Assignment          0
weekly_lab_quizzes_100                0
ass_ex_9                              0
ass_ex_8                              0
ass_ex_7                              0
ass_ex_6                              0
ass_ex_5                              0
ass_ex_3                              0
active_days                           0
ass_ex_2                              0
ass_ex_10                             0
ass_ex_1                              0
ass_2_with_penalty_100                0
ass_1_with_penalty_100                0
avg_events_per_active_day             0
weekend_events                        0
dtype: int64

In [17]:
X_aca_train.describe().T

,count,mean,std,min,25%,50%,75%,max
total_events,99.0,333.505051,264.162534,0.0,159.500000,266.000000,448.00000,1292.000000
active_days,99.0,41.151515,24.222339,0.0,24.000000,38.000000,55.50000,123.000000
unique_materials,99.0,25.424242,11.087860,0.0,17.000000,25.000000,33.00000,56.000000
after_hours_events,99.0,137.383838,132.452973,0.0,59.000000,98.000000,173.00000,851.000000
weekend_events,99.0,66.161616,66.554674,0.0,24.000000,45.000000,86.00000,405.000000
avg_events_per_active_day,99.0,7.630080,2.946773,0.0,5.868056,7.372093,9.44699,18.366667
ass_1_with_penalty_100,99.0,58.530303,29.586851,0.0,43.000000,64.000000,81.50000,100.000000
ass_2_with_penalty_100,99.0,54.287374,32.231676,0.0,34.425000,62.750000,81.25000,97.900000
ass_ex_1,99.0,66.666667,35.927932,0.0,45.000000,80.000000,100.00000,100.000000
ass_ex_10,99.0,44.898990,44.607013,0.0,0.000000,45.000000,90.00000,100.000000
